# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'F\x84Z\xf2U\xea\x19\xe6%\x19_l\x1d\x8ck\xdbC\x9e\x91\x10\xc0m\x893\x13\xb7\xede\xd0\xce+\xa3\x0f4<l\xed\x1ea\xc8x\xd8\r\xbe\x08/\xc4)\x95\xd6\xff4U\r8\xaf"\x00XQ \xa1Z('

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b'}\xdd\x17\\m\xc8E\x810\xaa\xd0\x9ai$\xbfJ\xdce\xc2j@\xa6\x8a+\xfd\xc4\xaf1\x14y\xb1=\x13\xa5\xa3\x99\xc51\xdcm\x95]\xa5\xbf\xd2\x11\xec\xe5\xbb\xb1\x98\x0bH\x96\x15\xae/\xaa\xdc@\xd9\xb6\xed\xb0' over socket

-   Latency and throughput determined by two factors
    1.  How long to slice the chunk from `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000888997 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000154 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b'\xbb\xa5\xee\xf6\xedn\xb7\xdb\xca\xda\x16X\xa6\x8d\x896z\xfc\x87\xdc\x14\x99\xe3\x1e\x16\xf4\xe6\x14\xc1i\xbc\x7f\x9d\x08\xbc\xef\x00z\xb3\xba\xd4\xa5\x01\x1b\xaa]\xd0P\xaa\xe0s\x96,\xe6\xe8E@B\xd0)c\xe0\x9f\xd2o\xad\x08\xae\x04\xbb0\xb6g\xea\x90D\x19_\t\x84\xfc\x9a\xf7a8z\xae-\x01\xbd\x919\xa3\x13n\x98\x05\x94\x93\xf3\xa5\x11\xba\xac\x84#t\x9b\x1b-}\x983\xe7g\xa12\x08\xa9\x8eQ{\xf7G\xf79XR[\xb4\x7f\x18\xb6\xc3H\xd8tA1\x8a\x92<6\x04\xfdE3\x9e\xcdj\xf2\xac\xb1\xfe\xc2\xfc\xf2\x8e\xbe\x9e\x91\n\xba\x08\x17\xec\x84\x96\xed\xf7]\xd83y;-\xb7\xec\x90z\xc8m\xae\x90p\xf7Q\x8b\x02\xa9D\xd5^-!\x1f\x1c\\\x85\x107\x18\xffl\x87<U\xc5f\xb0\xb6\x8c\xdf\x87\xec\xe0;\xf4}P\x91\xba\xb6\xb8\xca\xae\xaf\xe2\xc2j\xf0\xc7\xa0Y\xad\x94\xfc)\xeb\xd2s\xdc\xf6\xba\x81\xc0H\x7f\x1a\xda\x98\x0e\xdc\xa6T\xa8Z\x98\xa0\xbf\x90\xa6\xbe\x1e\xe8\x1f\xf4\xd5\xa6K\xec\x1fx!\xf2qV\xb3\xeb}\xa4E\x98\xab\xd2-\x8b\xcb\x9c\xde\xa7[\xfdC\xff\x97\xfc_\xbal\xe6\xce\xda\x00\xa8P;\xea\xc2\xdf\x86\xea

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.000874510 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'|d(\t\x88\xa7Wx\xc2\r\xeaa\x90g\x93X\xd8i\xa2\x18W\xe6\xd2n\xfa\xe4\x01V\x10y\xb4\xdb\x83%\x06N\x97\xfc\xf2\x8c\xc9\xaaH\x1b\xc9\x91[4\xee\x897\xa7\x99\xa4\xdbk\xa6z\xae+\xcf/k\xba\xa5\xcb\xb9\x04=\x9e6\xc2\x9c\x87\x8a\x02\x13\xa1yp\x1ad\xe8\xa8,\xc1vu#\x90\xa2yV\xec3\xe7\x95\x18\x1b\xc6\xac\'\x83\x95\x04\x8e"Bq\xa4\x01\xf3\xd2\xd6\x02l\x90\x95\xdb]\r\x9f\xca)\xe6\xdb\x0f^\xfb\xf6G\xf1\xd7\xe1\xe2\x7f\x18\xda\x10\xa7_\x12\xff\xeb\x85n%\x00h\xca}\xf8\x1c:\xafnf\xcf\x9aJ`_\t\x88b\x05\xaf\xcf\xd0\xf6n=\xe9\xef#\xaf\x98Ge\x08y2mHn\x12\xb0\x17\xcbF-yvK\xbe\xcd_^\x1d\xb9N\xb3\x83yBV\x7fT\x88\r\xf6n\x86\x16\xfe\x8fERG\xf4~>!\x84_\xed]\xe9\xe5\xe3\xe1\x1b\xaex\xa0y\xc2\x94F#{\x1a\x81\xca\x1c\x88KU>\xf8\x0bfTZ\xfa\xd1\xd3\xef\x15++\xec\x895\r\xa8\x1a.\xe6*\xd3\xeeQZ\x8fK\x85\xc7\xd4%\x0c\xc4\x8a\xe0\xd7m\x02\xc3\xf8/&E\x92\x1f}S\xb9\xd5P\xb4\x94L\r\x18w\\\xe4V\xd5L)\xe4\x93\x88N\x89p\xd1\xf1\xec'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000025411 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies